In [1]:
import os
import pandas as pd
from datetime import datetime
import importlib
import numpy as np

np.set_printoptions(suppress=True)

In [2]:
# Relative imports
os.chdir("..")
import hidden_state_model.processor
importlib.reload(hidden_state_model.processor)
Processor = hidden_state_model.processor.Processor
os.chdir("hidden_state_model")

### Read (and compact) dataframes

In [3]:
compact = True

In [4]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
for file in os.listdir("data"):
    if file.endswith(".parquet"):
        read.append(file)
        print(f"Reading {file}")
        df = pd.read_parquet(f"data/{file}")
        dfs.append(df)
    if file.endswith(".csv"):
        read.append(file)
        df = pd.read_csv(f"data/{file}", index_col=0)
        dfs.append(df)

raw_df = pd.concat(dfs)

if compact and len(dfs) > 10:
    print("Compacintg dfs")

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    raw_df.to_parquet(f"data/compacted_{timestamp}.parquet")
    
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    trash = "data/trash"
    for f in read:
        os.rename(f"data/{f}", f"{trash}/{f}")

dfs = []  # Clear memory
raw_df

Reading 2024-09-03_18-52-01.parquet
Reading 2024-09-03_22-10-59.parquet
Reading 2024-09-03_22-16-27.parquet
Reading 2024-09-03_20-05-30.parquet
Reading 2024-09-03_22-15-50.parquet
Reading 2024-09-03_19-00-58.parquet
Reading 2024-09-03_22-17-59.parquet
Reading 2024-09-03_18-58-04.parquet
Reading 2024-09-03_19-22-21.parquet
Reading 2024-09-03_22-17-34.parquet
Reading compacted_20240903175658.parquet
Reading 2024-09-03_22-08-39.parquet
Reading 2024-09-03_18-53-24.parquet
Compacintg dfs


,prev_entry,public_cards,player_piles,current_player_i,bet_in_stage,bet_in_game,player_has_played,player_is_folded,first_better_i,big_blind,player_name,player_type,action,amount,p,relative_ev,rank,tiebreakers,hand_index,state_id
state_id,,,,,,,,,,,,,,,,,,,,
bb1e32ea-283e-443e-b497-bf319b223f54,None,[],"[98, 96, 100]",0,"[2, 4, 0]","[2, 4, 0]","[False, False, True]","[False, False, True]",0,4,Tord,HumanPlayer,call,2,0.5399,0.010798,0,"[11, 3, 0, 0, 0]",536.0,NaN
031ffc62-99b3-4738-9b13-de72d818be93,None,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,call,2,0.7384,0.022152,1,"[7, 0, 0, 0, 0]",1167.0,NaN
09def076-8ba4-44d7-8b78-1362c3e6d5ed,031ffc62-99b3-4738-9b13-de72d818be93,"[38, 44, 27]","[96, 96]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.7507,0.030028,1,"[7, 12, 5, 1, 0]",1167.0,NaN
7e10f5b4-2c5b-449e-99e2-1589a9adfffd,09def076-8ba4-44d7-8b78-1362c3e6d5ed,"[38, 44, 27, 47]","[92, 92]",0,"[0, 0]","[8, 8]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,8,0.6928,0.055424,1,"[7, 12, 8, 5, 0]",1167.0,NaN
00536a74-872d-4e7c-b15b-d9c716566824,7e10f5b4-2c5b-449e-99e2-1589a9adfffd,"[38, 44, 27, 47, 9]","[84, 84]",0,"[0, 0]","[16, 16]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,check,0,0.6289,0.100624,1,"[7, 12, 9, 8, 0]",1167.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
e0f490ba-8a1a-4bcf-bb56-cd369dd1f0f7,None,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,call,2,0.4547,0.013641,0,"[8, 0, 0, 0, 0]",878.0,NaN
119c6619-1e91-4e20-8dab-fb59a2e700b2,None,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,call,2,0.5732,0.017196,0,"[11, 4, 0, 0, 0]",750.0,NaN
744f9975-54d5-4710-9d0b-a71e6256cf7d,119c6619-1e91-4e20-8dab-fb59a2e700b2,"[47, 33, 41]","[96, 96]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,check,0,0.4337,0.017348,0,"[11, 8, 7, 4, 2]",750.0,NaN


In [5]:
raw_df.dtypes

prev_entry            object
public_cards          object
player_piles          object
current_player_i       int64
bet_in_stage          object
bet_in_game           object
player_has_played     object
player_is_folded      object
first_better_i         int64
big_blind              int64
player_name           object
player_type           object
action                object
amount                 int64
p                    float64
relative_ev          float64
rank                   int64
tiebreakers           object
hand_index           float64
state_id             float64
dtype: object

In [6]:
# Check for conflicting rows
dupe_df = raw_df[raw_df.index.duplicated()]
assert len(dupe_df) == 0, dupe_df

## Process data

In [7]:
processor = Processor(raw_df)
df = processor.get_processed_df()
df

,game_id,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,...,opponent_check_river,opponent_check_showdown,action,amount,excess_rank,p,relative_ev,stage,player_name,n_players
bb1e32ea-283e-443e-b497-bf319b223f54,00b27a71-8249-4fa4-9350-577739433a85,0,0,0,0,0,0,0,0,0,...,0,0,call,2,0,0.5399,0.010798,preflop,Tord,2
031ffc62-99b3-4738-9b13-de72d818be93,0742cc68-af57-4cfc-ab37-7c2563dbc84b,0,0,0,0,0,0,0,0,0,...,0,0,call,2,1,0.7384,0.022152,preflop,Tord,2
09def076-8ba4-44d7-8b78-1362c3e6d5ed,0742cc68-af57-4cfc-ab37-7c2563dbc84b,0,0,0,0,0,2,0,0,0,...,0,0,raise,4,1,0.7507,0.030028,flop,Tord,2
7e10f5b4-2c5b-449e-99e2-1589a9adfffd,0742cc68-af57-4cfc-ab37-7c2563dbc84b,0,4,0,0,0,2,0,0,0,...,0,0,raise,8,1,0.6928,0.055424,turn,Tord,2
00536a74-872d-4e7c-b15b-d9c716566824,0742cc68-af57-4cfc-ab37-7c2563dbc84b,0,4,8,0,0,2,0,0,0,...,0,0,check,0,1,0.6289,0.100624,river,Tord,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
e0f490ba-8a1a-4bcf-bb56-cd369dd1f0f7,dc6de31a-d186-419c-88bb-4c47fc75c3e0,0,0,0,0,0,0,0,0,0,...,0,0,call,2,0,0.4547,0.013641,preflop,Tord,2
119c6619-1e91-4e20-8dab-fb59a2e700b2,96edfd88-4927-46fa-a4ac-051c0313de47,0,0,0,0,0,0,0,0,0,...,0,0,call,2,0,0.5732,0.017196,preflop,Tord,2
744f9975-54d5-4710-9d0b-a71e6256cf7d,96edfd88-4927-46fa-a4ac-051c0313de47,0,0,0,0,0,2,0,0,0,...,0,0,check,0,0,0.4337,0.017348,flop,Tord,2
70552eec-91ff-4080-9280-bfc0d9d33530,17679412-8439-4cd4-86cd-d6d68621bbe1,0,0,0,0,0,0,0,0,0,...,0,0,call,2,0,0.4452,0.013356,preflop,Tord,2


In [8]:
df.dtypes

game_id                     object
raise_preflop                int64
raise_flop                   int64
raise_turn                   int64
raise_river                  int64
raise_showdown               int64
call_preflop                 int64
call_flop                    int64
call_turn                    int64
call_river                   int64
call_showdown                int64
check_preflop                int64
check_flop                   int64
check_turn                   int64
check_river                  int64
check_showdown               int64
opponent_raise_preflop       int64
opponent_raise_flop          int64
opponent_raise_turn          int64
opponent_raise_river         int64
opponent_raise_showdown      int64
opponent_call_preflop        int64
opponent_call_flop           int64
opponent_call_turn           int64
opponent_call_river          int64
opponent_call_showdown       int64
opponent_check_preflop       int64
opponent_check_flop          int64
opponent_check_turn 

## Training

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [10]:
X = df.drop(["excess_rank", "game_id", "p", "relative_ev"], axis=1)
y = df["excess_rank"]
groups = df["game_id"]  # Group by 'game_id' to ensure no data leakage

In [11]:
X

,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,call_showdown,...,opponent_check_preflop,opponent_check_flop,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,stage,player_name,n_players
bb1e32ea-283e-443e-b497-bf319b223f54,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,call,2,preflop,Tord,2
031ffc62-99b3-4738-9b13-de72d818be93,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,call,2,preflop,Tord,2
09def076-8ba4-44d7-8b78-1362c3e6d5ed,0,0,0,0,0,2,0,0,0,0,...,0,1,0,0,0,raise,4,flop,Tord,2
7e10f5b4-2c5b-449e-99e2-1589a9adfffd,0,4,0,0,0,2,0,0,0,0,...,0,1,0,0,0,raise,8,turn,Tord,2
00536a74-872d-4e7c-b15b-d9c716566824,0,4,8,0,0,2,0,0,0,0,...,0,1,0,0,0,check,0,river,Tord,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
e0f490ba-8a1a-4bcf-bb56-cd369dd1f0f7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,call,2,preflop,Tord,2
119c6619-1e91-4e20-8dab-fb59a2e700b2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,call,2,preflop,Tord,2
744f9975-54d5-4710-9d0b-a71e6256cf7d,0,0,0,0,0,2,0,0,0,0,...,0,1,0,0,0,check,0,flop,Tord,2
70552eec-91ff-4080-9280-bfc0d9d33530,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,call,2,preflop,Tord,2


In [12]:
y

bb1e32ea-283e-443e-b497-bf319b223f54    0
031ffc62-99b3-4738-9b13-de72d818be93    1
09def076-8ba4-44d7-8b78-1362c3e6d5ed    1
7e10f5b4-2c5b-449e-99e2-1589a9adfffd    1
00536a74-872d-4e7c-b15b-d9c716566824    1
                                       ..
e0f490ba-8a1a-4bcf-bb56-cd369dd1f0f7    0
119c6619-1e91-4e20-8dab-fb59a2e700b2    0
744f9975-54d5-4710-9d0b-a71e6256cf7d    0
70552eec-91ff-4080-9280-bfc0d9d33530    0
50a62bfb-2c21-4e83-a07c-6a0e4528a22b    0
Name: excess_rank, Length: 4080, dtype: int64

In [13]:
# Identify categorical columns (excluding 'game_id')
categorical_cols = ["action", "stage", "player_name"]

# Preprocessing pipeline: OneHotEncoding for categorical and scaling for numerical
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first"), categorical_cols)],
    remainder="passthrough",
)

# Create the full pipeline with logistic regression
model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial", solver="lbfgs", max_iter=10_000
            ),
        ),
    ]
)

In [14]:
# Grouped train-test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (3266, 35)
Test shape: (814, 35)


In [15]:
# Train the model
model.fit(X_train, y_train)

# Evaluate the model
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy}")

Accuracy: 0.7764127764127764


In [16]:
probabilities = model.predict_proba(X_test)
prob_df = pd.DataFrame(probabilities, columns=model.classes_, index=X_test.index)
prob_df["true"] = y_test.values
prob_df["pred"] = model.predict(X_test)
prob_df["correct"] = prob_df["true"] == prob_df["pred"]
prob_df["goodness"] = prob_df.apply(lambda x: x.get(x["true"]) or 0, axis=1)
print("Accuracy", prob_df["correct"].mean())
print("Mean goodness: ", prob_df["goodness"].mean())
prob_df

Accuracy 0.7764127764127764
Mean goodness:  0.691950314094859


,0,1,2,3,4,5,true,pred,correct,goodness
031ffc62-99b3-4738-9b13-de72d818be93,0.959640,0.039591,0.000111,0.000087,0.000142,0.000429,1,0,False,0.039591
09def076-8ba4-44d7-8b78-1362c3e6d5ed,0.135854,0.755646,0.086773,0.007425,0.008715,0.005587,1,1,True,0.755646
7e10f5b4-2c5b-449e-99e2-1589a9adfffd,0.059910,0.725262,0.182847,0.007880,0.014163,0.009938,1,1,True,0.725262
00536a74-872d-4e7c-b15b-d9c716566824,0.154888,0.756360,0.030466,0.011190,0.022627,0.024468,1,1,True,0.756360
1b7a2171-6b7d-4dad-86c3-abbfb4d4070f,0.222417,0.743376,0.014718,0.000008,0.006862,0.012619,1,1,True,0.743376
...,...,...,...,...,...,...,...,...,...,...
aeea29ac-6dbc-4feb-a9eb-a6f90518c077,0.959640,0.039591,0.000111,0.000087,0.000142,0.000429,0,0,True,0.959640
b83c8b26-0647-4c08-80a7-73caf62b225f,0.959640,0.039591,0.000111,0.000087,0.000142,0.000429,0,0,True,0.959640
5fb22d35-1368-4d77-a1fd-da3d10e45a79,0.772790,0.214304,0.005559,0.002358,0.002323,0.002666,0,0,True,0.772790
11f68e36-dd1c-4fab-8f70-ddbc9a452bd5,0.750018,0.213667,0.005090,0.008298,0.013831,0.009096,0,0,True,0.750018


In [17]:
# Look at incorrect predictions
print(prob_df[prob_df["correct"] == False].shape[0], "incorrect predictions:")
prob_df[prob_df["correct"] == False]

182 incorrect predictions:


,0,1,2,3,4,5,true,pred,correct,goodness
031ffc62-99b3-4738-9b13-de72d818be93,0.959640,0.039591,0.000111,0.000087,1.421325e-04,0.000429,1,0,False,0.039591
b0874bf3-b9c2-4e92-a437-c36f24db6b7b,0.567172,0.403910,0.006188,0.003018,2.858699e-03,0.016854,1,0,False,0.403910
5400fa30-f3b6-4b75-86ec-732a71c8add0,0.099226,0.713306,0.131718,0.020089,1.127621e-02,0.024384,2,1,False,0.131718
7585d2ae-ab24-4c20-883b-43eb6a0e78c0,0.396886,0.524609,0.016790,0.004237,6.933275e-03,0.050545,2,1,False,0.016790
556db197-9d65-4564-876b-48ab3b6a3fe9,0.028586,0.466846,0.097691,0.012353,8.262095e-02,0.311902,2,1,False,0.097691
...,...,...,...,...,...,...,...,...,...,...
6098209472,0.545472,0.395913,0.032145,0.009178,1.366165e-02,0.003631,1,0,False,0.395913
6100754688,0.495173,0.478374,0.014792,0.005682,4.459277e-03,0.001520,1,0,False,0.478374
6145168192,0.180731,0.589896,0.212805,0.000110,1.377420e-07,0.016459,0,1,False,0.180731
6146897664,0.220735,0.621010,0.122802,0.000230,2.836776e-07,0.035222,0,1,False,0.220735


### Attempt to infer card probabilities from rank and table

In [18]:
from cpp_poker.cpp_poker import Hand, CardCollection, HandRank, CheatSheet

In [19]:
hand_ranks_per_state = []
table_ranks = []
for i, (state_id, row) in enumerate(prob_df.iterrows()):
    raw_row = raw_df.loc[state_id]
    table_cards = CardCollection(raw_row["public_cards"])
    table_rank = table_cards.rank_hand().get_rank()
    excess_hand_ranks_row = table_cards.rank_all_hands()
    hand_ranks_per_state.append(
        [rank.get_rank() - table_rank for rank in excess_hand_ranks_row]
    )
    table_ranks.append(table_rank)

table_ranks_df = pd.DataFrame(table_ranks, index=prob_df.index, columns=["table_rank"])
hand_ranks_df = pd.DataFrame(hand_ranks_per_state, index=prob_df.index)
hand_ranks_df

,0,1,2,3,4,5,6,7,8,9,...,1316,1317,1318,1319,1320,1321,1322,1323,1324,1325
6152138816,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6152372176,1,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
6064222992,1,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
6152371360,1,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
6152139824,1,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6102526320,1,1,1,1,1,1,1,1,2,2,...,0,0,5,5,0,0,0,0,0,8
6127312464,1,1,1,2,1,1,1,1,2,2,...,5,5,5,5,0,5,5,5,5,8
6127438960,1,1,1,2,1,2,1,1,2,2,...,5,5,5,5,0,5,5,5,5,8
6431193840,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [20]:
# Get the values of the column indices from hand_ranks_df
column_indices = hand_ranks_df.values

# Use np.arange to create an array of row indices
row_indices = np.arange(hand_ranks_df.shape[0])[:, None]

# Use the row and column indices to index into the DataFrame values
hand_probs_df = pd.DataFrame(
    prob_df.values[row_indices, column_indices],
    index=hand_ranks_df.index,
    columns=hand_ranks_df.columns,
)

# Normalize the hand probabilities to sum to 1 for each row
hand_probs_df = hand_probs_df.div(hand_probs_df.sum(axis=1), axis=0)
hand_probs_df

,0,1,2,3,4,5,6,7,8,9,...,1316,1317,1318,1319,1320,1321,1322,1323,1324,1325
6152138816,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,...,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768,0.000768
6152372176,0.000785,0.000785,0.00077,0.00077,0.00077,0.00077,0.00077,0.00077,0.000785,0.00077,...,0.00077,0.00077,0.00077,0.00077,0.00077,0.00077,0.00077,0.00077,0.00077,0.00077
6064222992,0.000894,0.000894,0.000712,0.000712,0.000712,0.000712,0.000712,0.000712,0.000894,0.000712,...,0.000712,0.000712,0.000712,0.000712,0.000712,0.000712,0.000712,0.000712,0.000712,0.000712
6152371360,0.000958,0.000958,0.000688,0.000688,0.000688,0.000688,0.000688,0.000688,0.000958,0.000688,...,0.000688,0.000688,0.000688,0.000688,0.000688,0.000688,0.000688,0.000688,0.000688,0.000688
6152139824,0.001098,0.001098,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582,0.001098,0.000582,...,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6102526320,0.000361,0.000361,0.000361,0.000361,0.000361,0.000361,0.000361,0.000361,0.000003,0.000003,...,0.001068,0.001068,0.000017,0.000017,0.001068,0.001068,0.001068,0.001068,0.001068,0.001469
6127312464,0.000679,0.000679,0.000679,0.000005,0.000679,0.000679,0.000679,0.000679,0.000005,0.000005,...,0.000076,0.000076,0.000076,0.000076,0.001622,0.000076,0.000076,0.000076,0.000076,0.0
6127438960,0.000663,0.000663,0.000663,0.000003,0.000663,0.000003,0.000663,0.000663,0.000003,0.000003,...,0.000146,0.000146,0.000146,0.000146,0.001784,0.000146,0.000146,0.000146,0.000146,0.0
6431193840,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,...,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781,0.000781


In [21]:
# Validate that rows sum to 1
hand_probs_df.sum(axis=1)

6152138816    1.0
6152372176    1.0
6064222992    1.0
6152371360    1.0
6152139824    1.0
             ... 
6102526320    1.0
6127312464    1.0
6127438960    1.0
6431193840    1.0
6127534384    1.0
Length: 753, dtype: object

In [22]:
# Look at the max probabilites for different hands
mask = prob_df["pred"] == 2
max_probs = hand_probs_df[mask].max(axis=1)
min_probs = hand_probs_df[mask].min(axis=1)
max_prob_hands = hand_probs_df[mask].idxmax(axis=1)
min_prob_hands = hand_probs_df[mask].idxmin(axis=1)
mean_probs = hand_probs_df[mask].mean(axis=1)
sample_prob_df = pd.DataFrame(
    {
        "max_prob": max_probs,
        "max_prob_hand": max_prob_hands,
        "min_prob": min_probs,
        "min_prob_hand": min_prob_hands,
        "mean_prob": mean_probs,
    }
)
sample_prob_df["max_prob_hand"] = sample_prob_df["max_prob_hand"].apply(lambda x: Hand(x).str())
sample_prob_df["min_prob_hand"] = sample_prob_df["min_prob_hand"].apply(lambda x: Hand(x).str())
sample_prob_df["table"] = raw_df.loc[sample_prob_df.index, "public_cards"].apply(
    lambda x: CardCollection(x).str()
)
sample_prob_df["predicted_excess_rank"] = prob_df.loc[sample_prob_df.index, "pred"]
sample_prob_df["pred_rank"] = (
    sample_prob_df["predicted_excess_rank"]
    + table_ranks_df.loc[sample_prob_df.index, "table_rank"]
)
sample_prob_df["pred_rank_str"] = sample_prob_df["pred_rank"].apply(
    lambda x: HandRank(x, []).get_rank_name()
)
sample_prob_df = sample_prob_df.drop(["predicted_excess_rank"], axis=1)
sample_prob_df

,max_prob,max_prob_hand,min_prob,min_prob_hand,mean_prob,table,pred_rank,pred_rank_str
6064224576,0.000875,♥ 2 ♥ 3,0.000129,♥ 2 ♥ A,0.000754,♥ 7 ♦ 7 ♣ 7 ♠ A,5,Flush
6064257536,0.005945,♥ 2 ♠ 7,0.000499,♥ 2 ♥ 3,0.000754,♥ 7 ♦ 7 ♣ 7 ♠ A,5,Flush
5803868512,0.002783,♥ 5 ♥ 6,0.000002,♥ 2 ♥ 3,0.000754,♥ 4 ♦ J ♣ 5 ♣ 6,2,Two Pairs
6165563904,0.003191,♥ 9 ♥ J,0.000042,♦ 2 ♦ 3,0.000754,♦ 9 ♦ J ♦ A,2,Two Pairs
6085606656,0.002888,♥ 2 ♥ 5,0.000022,♥ 2 ♦ 2,0.000754,♥ 9 ♦ 5 ♣ 2 ♣ K,2,Two Pairs
6087820048,0.004509,♥ 2 ♥ 5,0.000012,♥ 2 ♦ 2,0.000754,♥ 9 ♦ 5 ♣ 2 ♣ K,2,Two Pairs
6125188576,0.001414,♥ 7 ♥ 8,0.000004,♥ 7 ♦ 7,0.000754,♥ 3 ♥ 9 ♦ 8 ♠ 7,2,Two Pairs


In [23]:
# Deepdive into a specific row
state_id = "6165563904"
row = sample_prob_df.loc[state_id]
most_likely_hands = hand_probs_df.loc[state_id].sort_values(ascending=False).head(5)
least_likely_hands = hand_probs_df.loc[state_id].sort_values(ascending=True).head(5)
print("Table cards:", row["table"])
print("Predicted excess rank:", prob_df.loc[state_id, "pred"])
print("Actual excess rank", df.loc[state_id, "excess_rank"])
print("Most and least likely hands:")
for hand, prob in most_likely_hands.items():
    print(f"{Hand(hand).str()}: {prob*100:.2f}%")
print("...")
for hand, prob in least_likely_hands.items():
    print(f"{Hand(hand).str()}: {prob*100:.2f}%")

Table cards: ♦ 9 ♦ J ♦ A 
Predicted excess rank: 2
Actual excess rank 1
Most and least likely hands:
♣ 9 ♠ J : 0.32%
♥ A ♠ J : 0.32%
♣ A ♠ J : 0.32%
♣ 9 ♠ A : 0.32%
♥ A ♣ 9 : 0.32%
...
♦ 4 ♦ 7 : 0.00%
♦ 7 ♦ K : 0.00%
♦ 10 ♦ K : 0.00%
♦ 10 ♦ Q : 0.00%
♦ 4 ♦ Q : 0.00%


### Attempt to infer winning probabilities from hidden state probabilities

In [24]:
hand_winning_probs = []
for i, (state_id, row) in enumerate(hand_probs_df.iterrows()):
    print(f"Processing row {i}", flush=True)
    raw_row = raw_df.loc[state_id]
    table_cards = CardCollection(raw_row["public_cards"])
    processed_row = df.loc[state_id]
    n_players = processed_row["n_players"]
    hand_winning_probs.append(CheatSheet.get_all_winning_probabilities(table_cards, n_players, 1000))

hand_winning_probs_df = pd.DataFrame(hand_winning_probs, index=hand_probs_df.index)
hand_winning_probs_df

Processing row 0
